<a href="https://colab.research.google.com/github/mobius29er/AIML_Class/blob/main/colab_activity_21_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Colab Activity 21.6: Hyperparameter Tuning with Keras

**Expected Time = 60 minutes**



This activity focuses on using hyperparameter tuning with the `keras` library.  There are two ways to perform a grid search with `keras`, and you will implement both. While `keras_tuner` was discussed in the lectures, here you will use the `Scikit-Learn` wrapper for keras to grid search the parameters using `GridSearchCV`.  You will implement this with the `KerasClassifier` to build some basic models on the wine dataset.  

#### Index

- [Problem 1](#-Problem-1)
- [Problem 2](#-Problem-2)
- [Problem 3](#-Problem-3)
- [Problem 4](#-Problem-4)

In [1]:
!pip uninstall scikit-learn -y
!pip install scikit-learn==1.3.2

Found existing installation: scikit-learn 1.7.2
Uninstalling scikit-learn-1.7.2:
  Successfully uninstalled scikit-learn-1.7.2
  Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (10.8 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikeras 0.13.0 requires scikit-learn>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.3.2 which is incompatible.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, but you have scikit-learn 1.3.2 which is incompatible.


In [2]:
!pip install scikeras

  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.utils import to_categorical

### The Data

Below, the wine dataset is loaded, split, and scaled.  

In [4]:
wine = load_wine(as_frame=True)

In [5]:
wine.frame.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [6]:
wine.frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  targe

In [7]:
X = wine.data
y = to_categorical(wine.target)

In [8]:
X_scaled = StandardScaler().fit_transform(X)

[Back to top](#-Index)

### Problem 1

#### The Build Function



To use the `KerasClassifier` you first need to write a function that creates a `keras` model and takes in arguments for the parameters you wish to search. The pseudocode for this function is given below:

```python
def create_model(optimizer=..., neurons=..., activation=..., input_dim=...):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(neurons, activation=activation, input_shape=(input_dim,)))
    model.add(tf.keras.layers.Dense(1))  # Output layer for regression
    model.compile(optimizer=..., loss=....)
    return model
```

Your goal is to complete the definition of the `create_model` function using the arguments `optimizer = 'adam'` and `neurons=50` for `activation = 'relu'` and `input_dim = 13`. Inside the function, compile the model using the selected `optimizer` and `loss= 'mse'`.



In [9]:
tf.random.set_seed(42)
# Function to create a fully connected neural network model for SciKeras
def create_model(optimizer='adam', neurons=50, activation="relu", input_dim=13):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(neurons, activation=activation, input_shape=(input_dim,)))
    model.add(tf.keras.layers.Dense(1))  # Output layer for regression
    model.compile(optimizer=optimizer, loss='mse')
    return model

[Back to top](#-Index)

### Problem 2

#### Creating the `KerasRegressos` model



Now, use the `create_model` function to instantiate `KerasRegressor` as `model` with  `verbose = 2`.


In [10]:
# Keras model with SciKeras wrapper
model = KerasRegressor(model = create_model, verbose=2)

[Back to top](#-Index)

### Problem 3

#### Performing the Grid Search



Now, to perform a grid search you just need to create a dictionary named `param_grid` with the hyperparameter `'model__neurons' : [10, 50, 100]`,     `'model__activation': ['relu', 'sigmoid']`,
`'model__optimizer': ['adam', 'sgd']`, `'batch_size': [1, 10]`, and
`'epochs': [10, 20]`.  

In [11]:
tf.random.set_seed(42)
# Hyperparameters to be optimized
param_grid = {
  'model__neurons' : [10, 50, 100],
  'model__activation': ['relu', 'sigmoid'],
  'model__optimizer': ['adam', 'sgd'],
  'batch_size': [1, 10],
  'epochs': [10, 20]
}

[Back to top](#-Index)

### Problem 4

#### Fit and Evaluate the model



Use the `GridSearchCV` function with `estimator=model`, `param_grid=param_grid`, `scoring='neg_mean_squared_error'`, `cv=3`, and `verbose=2` to search your parameters and assign the results to `grid`. Next, use function `fit` on `grid` with the training data to fit your model.

In [12]:
# GridSearchCV for hyperparameter tuning
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3, verbose=2)
grid_result = grid.fit(X_scaled, y)

# Display the best hyperparameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


Fitting 3 folds for each of 48 candidates, totalling 144 fits
Epoch 1/10
118/118 - 1s - 11ms/step - loss: 1.3054
Epoch 2/10
118/118 - 0s - 2ms/step - loss: 0.6475
Epoch 3/10
118/118 - 0s - 2ms/step - loss: 0.5025
Epoch 4/10
118/118 - 0s - 2ms/step - loss: 0.4250
Epoch 5/10
118/118 - 0s - 2ms/step - loss: 0.3734
Epoch 6/10
118/118 - 0s - 2ms/step - loss: 0.3379
Epoch 7/10
118/118 - 0s - 2ms/step - loss: 0.3136
Epoch 8/10
118/118 - 0s - 2ms/step - loss: 0.2968
Epoch 9/10
118/118 - 0s - 2ms/step - loss: 0.2850
Epoch 10/10
118/118 - 0s - 2ms/step - loss: 0.2760
60/60 - 0s - 4ms/step
[CV] END batch_size=1, epochs=10, model__activation=relu, model__neurons=10, model__optimizer=adam; total time=   4.9s
Epoch 1/10
119/119 - 1s - 10ms/step - loss: 1.2622
Epoch 2/10
119/119 - 0s - 2ms/step - loss: 0.6823
Epoch 3/10
119/119 - 0s - 2ms/step - loss: 0.4864
Epoch 4/10
119/119 - 0s - 2ms/step - loss: 0.4064
Epoch 5/10
119/119 - 0s - 2ms/step - loss: 0.3647
Epoch 6/10
119/119 - 0s - 2ms/step - loss: 0

6/6 - 0s - 31ms/step
[CV] END batch_size=10, epochs=10, model__activation=relu, model__neurons=10, model__optimizer=sgd; total time=   2.0s
Epoch 1/10
12/12 - 1s - 80ms/step - loss: 0.7383
Epoch 2/10
12/12 - 0s - 5ms/step - loss: 0.5543
Epoch 3/10
12/12 - 0s - 6ms/step - loss: 0.4652
Epoch 4/10
12/12 - 0s - 6ms/step - loss: 0.4113
Epoch 5/10
12/12 - 0s - 6ms/step - loss: 0.3754
Epoch 6/10
12/12 - 0s - 6ms/step - loss: 0.3500
Epoch 7/10
12/12 - 0s - 6ms/step - loss: 0.3311
Epoch 8/10
12/12 - 0s - 5ms/step - loss: 0.3166
Epoch 9/10
12/12 - 0s - 5ms/step - loss: 0.3052
Epoch 10/10
12/12 - 0s - 6ms/step - loss: 0.2960
6/6 - 0s - 55ms/step
[CV] END batch_size=10, epochs=10, model__activation=relu, model__neurons=10, model__optimizer=sgd; total time=   2.0s
Epoch 1/10
12/12 - 1s - 70ms/step - loss: 0.3515
Epoch 2/10
12/12 - 0s - 6ms/step - loss: 0.3076
Epoch 3/10
12/12 - 0s - 6ms/step - loss: 0.2951
Epoch 4/10
12/12 - 0s - 6ms/step - loss: 0.2870
Epoch 5/10
12/12 - 0s - 5ms/step - loss: 0.28

6/6 - 0s - 32ms/step
[CV] END batch_size=10, epochs=10, model__activation=relu, model__neurons=50, model__optimizer=adam; total time=   2.4s
Epoch 1/10
12/12 - 1s - 117ms/step - loss: 1.9676
Epoch 2/10
12/12 - 0s - 6ms/step - loss: 1.0711
Epoch 3/10
12/12 - 0s - 6ms/step - loss: 0.5823
Epoch 4/10
12/12 - 0s - 6ms/step - loss: 0.3709
Epoch 5/10
12/12 - 0s - 6ms/step - loss: 0.3055
Epoch 6/10
12/12 - 0s - 5ms/step - loss: 0.2894
Epoch 7/10
12/12 - 0s - 6ms/step - loss: 0.2814
Epoch 8/10
12/12 - 0s - 6ms/step - loss: 0.2741
Epoch 9/10
12/12 - 0s - 6ms/step - loss: 0.2681
Epoch 10/10
12/12 - 0s - 6ms/step - loss: 0.2634
6/6 - 0s - 60ms/step
[CV] END batch_size=10, epochs=10, model__activation=relu, model__neurons=50, model__optimizer=adam; total time=   2.5s
Epoch 1/10
12/12 - 1s - 115ms/step - loss: 0.5964
Epoch 2/10
12/12 - 0s - 6ms/step - loss: 0.3715
Epoch 3/10
12/12 - 0s - 6ms/step - loss: 0.3136
Epoch 4/10
12/12 - 0s - 6ms/step - loss: 0.2884
Epoch 5/10
12/12 - 0s - 5ms/step - loss: 

Finally, the results are written in a dataframe.

In [13]:

# Extract and display results from GridSearchCV
results = pd.DataFrame(grid_result.cv_results_)
print(results.head())

   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       4.001351      0.425783         0.332819        0.010057   
1       2.868971      0.049136         0.315585        0.002723   
2       3.571028      0.082875         0.329490        0.006061   
3       2.862525      0.007497         0.330265        0.001470   
4       3.701796      0.200177         0.346651        0.000747   

   param_batch_size  param_epochs param_model__activation  \
0                 1            10                    relu   
1                 1            10                    relu   
2                 1            10                    relu   
3                 1            10                    relu   
4                 1            10                    relu   

   param_model__neurons param_model__optimizer  \
0                    10                   adam   
1                    10                    sgd   
2                    50                   adam   
3                    50       

Because of the grading enviornment, a more exhaustive search over additional parameters is not an option.  To extend the work here should be straightforward enough, and this is a nice solution to grid searching the hyperparameters of a model.